# 🚀 Databricks Unity Catalog Setup & Troubleshooting

## 🎯 **Purpose**
This notebook helps you set up Unity Catalog properly by:
- ✅ Checking available catalogs in your workspace
- 🔧 Creating required catalogs (if permissions allow)
- 📊 Creating schemas for DEV, STAGING, and PROD environments
- 🚨 Handling common Unity Catalog errors

## ❌ **Common Error You Might See:**
```
[NO_SUCH_CATALOG_EXCEPTION] Catalog 'staging_digital_engineering_services' was not found
```

**💡 Solution:** Run this notebook step-by-step to fix catalog and schema setup!

In [ ]:
# Load environment variables
from dotenv import load_dotenv
import os
load_dotenv()

# Set environment for databricks-connect (fixed assignment issue)
if os.getenv('DATABRICKS_HOST') is not None: 
    os.environ['DATABRICKS_HOST'] = os.getenv('DATABRICKS_HOST')
if os.getenv('DATABRICKS_TOKEN') is not None: 
    os.environ['DATABRICKS_TOKEN'] = os.getenv('DATABRICKS_TOKEN')

# Safe print that handles None values
databricks_host = os.getenv('DATABRICKS_HOST')
if databricks_host:
    print(f"🔗 Connecting to: {databricks_host}")
else:
    print("🔗 Connecting to: [DATABRICKS_HOST not set]")

# Check if we have Databricks credentials
if not os.getenv('DATABRICKS_HOST') or not os.getenv('DATABRICKS_TOKEN'):
    print("❌ Missing Databricks credentials!")
    print("📝 In Databricks, environment variables aren't needed!")
    print("💡 This notebook should work directly in Databricks")
    print("🔑 Databricks automatically handles authentication when running on cluster")
    print("")
    print("✅ Proceeding with Databricks native authentication...")
else:
    print("✅ Databricks credentials loaded!")

# 📚 Import Required Libraries
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

try:
    from pyspark.sql import SparkSession
    from pyspark.sql.utils import AnalysisException
    
    # Initialize Spark Session (simplified for Databricks)
    print("🔄 Initializing Spark Session...")
    
    spark = SparkSession.builder.appName("MLOps-Unity-Catalog-Setup").getOrCreate()
    
    print("✅ Libraries imported successfully!")
    print(f"🔗 Spark version: {spark.version}")
    
    # Test connection with simple queries
    try:
        current_catalog = spark.sql('SELECT current_catalog()').collect()[0][0]
        current_schema = spark.sql('SELECT current_schema()').collect()[0][0]
        print(f"📊 Current catalog: {current_catalog}")
        print(f"🏪 Current schema: {current_schema}")
        print("🎉 Successfully connected to Databricks!")
    except Exception as e:
        print(f"⚠️  Connected to Spark but Unity Catalog access failed: {str(e)}")
        print("💡 This is normal if Unity Catalog isn't fully configured yet")
        
except Exception as e:
    print(f"❌ Failed to initialize Spark: {str(e)}")
    print("💡 Make sure this notebook is running on a Databricks cluster")
    spark = None

🔗 Connecting to: https://adb-269274943657929.9.azuredatabricks.net/
✅ Databricks credentials loaded!
🔄 Attempting to connect to Databricks cluster...
❌ Failed to connect to Databricks: [JAVA_GATEWAY_EXITED] Java gateway process exited before sending its port number.

💡 SOLUTIONS:
1. 🎯 RECOMMENDED: Upload this notebook to Databricks and run there
2. 🔧 Install Databricks Connect: pip install databricks-connect
3. 🔄 Configure cluster ID in Databricks Connect

For now, continuing with limited functionality...


## 🚨 **Local Execution Issue - Choose Your Approach**

### **The Problem:**
Unity Catalog operations need **Databricks cluster resources** - they can't run locally without proper Databricks Connect setup.

### **🎯 SOLUTION 1: Upload & Run in Databricks (EASIEST)**

**Steps:**
1. **Upload this notebook** to Databricks workspace
2. **Attach to cluster** `0330-192824-2qmica76`  
3. **Run all cells** - they'll work perfectly in Databricks

**How to upload:**
- Go to: https://adb-269274943657929.9.azuredatabricks.net/
- Navigate: **Workspace** → **Users** → **Your Name**
- Click **Import** → Upload `databricks_setup.ipynb`

---

### **🔧 SOLUTION 2: Fix Databricks Connect Locally**

**Install Databricks Connect:**

In [ ]:
# 🔧 Option 2: Install Databricks Connect for Local Development
print("🔧 Setting up Databricks Connect for local development...")
print("=" * 60)

# Check current environment
import sys
print(f"Python version: {sys.version}")
print(f"Current working directory: {os.getcwd()}")

# Instructions for Databricks Connect setup
setup_instructions = """
📦 INSTALL DATABRICKS CONNECT:
   1. pip uninstall pyspark  # Remove conflicting versions
   2. pip install databricks-connect==12.2.1  # Match your Databricks runtime
   3. databricks-connect configure --token
   
🔑 CONFIGURE ACCESS:
   Enter these values when prompted:
   - Host: https://adb-269274943657929.9.azuredatabricks.net/
   - Token: [Your personal access token]
   - Cluster ID: 0330-192824-2qmica76

⚡ ALTERNATIVE - Just Run in Databricks:
   Upload this notebook to Databricks and run there instead!
"""

print(setup_instructions)

# Check if user has proper .env setup
env_checks = []
required_vars = ['DATABRICKS_HOST', 'DATABRICKS_TOKEN']

print("\n🔍 CHECKING ENVIRONMENT SETUP:")
for var in required_vars:
    value = os.getenv(var)
    if value:
        if var == 'DATABRICKS_TOKEN':
            print(f"   ✅ {var}: {'*' * len(value[:10])}... (hidden)")
        else:
            print(f"   ✅ {var}: {value}")
        env_checks.append(True)
    else:
        print(f"   ❌ {var}: Not set")
        env_checks.append(False)

if all(env_checks):
    print("✅ Environment variables configured correctly!")
else:
    print("\n📝 Update your .env file with:")
    print("DATABRICKS_HOST=https://adb-269274943657929.9.azuredatabricks.net/")
    print("DATABRICKS_TOKEN=your-personal-access-token-here")
    
print("\n🎯 RECOMMENDED: Skip local setup and upload to Databricks instead!")

In [ ]:
# 🔍 Check Available Catalogs
print("🔍 Checking what catalogs are currently available in your workspace...")
print("=" * 60)

try:
    # Get list of existing catalogs
    catalogs_df = spark.sql("SHOW CATALOGS")
    available_catalogs = [row['catalog'] for row in catalogs_df.collect()]
    
    print(f"📋 Found {len(available_catalogs)} catalogs:")
    for i, catalog in enumerate(available_catalogs, 1):
        print(f"   {i}. {catalog}")
    
    # Check which catalogs we need
    required_catalogs = [
        "dev_digital_engineering_services",
        "staging_digital_engineering_services", 
        "prod_digital_engineering_services"
    ]
    
    print(f"\n🎯 Required catalogs for MLOps pipeline:")
    missing_catalogs = []
    for catalog in required_catalogs:
        if catalog in available_catalogs:
            print(f"   ✅ {catalog} - EXISTS")
        else:
            print(f"   ❌ {catalog} - MISSING")
            missing_catalogs.append(catalog)
    
    if missing_catalogs:
        print(f"\n⚠️  Found {len(missing_catalogs)} missing catalogs that need to be created!")
    else:
        print(f"\n🎉 All required catalogs already exist!")
        
    # Display the catalogs DataFrame for reference
    display(catalogs_df)
    
except Exception as e:
    print(f"❌ Error checking catalogs: {str(e)}")
    print("💡 You might need proper permissions to view catalogs.")

In [ ]:
# 🔧 Create Required Catalogs with Storage Location (If Missing & You Have Permissions)
print("🔧 Attempting to create missing catalogs with proper storage locations...")
print("=" * 60)

# Define catalogs with descriptions and storage paths
catalogs_to_create = {
    "dev_digital_engineering_services": "Development environment for MLOps water quality pipeline",
    "staging_digital_engineering_services": "Staging environment for MLOps water quality pipeline", 
    "prod_digital_engineering_services": "Production environment for MLOps water quality pipeline"
}

created_catalogs = []
failed_catalogs = []

print("💡 STORAGE LOCATION OPTIONS:")
print("   Option 1: Using DBFS managed storage (recommended)")
print("   Option 2: Using Volumes storage") 
print("   Option 3: Create via Databricks UI (easiest)")
print("")

for catalog_name, description in catalogs_to_create.items():
    try:
        print(f"\n🚀 Creating catalog: {catalog_name}")
        
        # Option 1: Try with DBFS managed location first
        storage_location = f"dbfs:/mnt/unity-catalog/{catalog_name}-storage"
        
        create_sql = f"""
        CREATE CATALOG IF NOT EXISTS {catalog_name} 
        MANAGED LOCATION '{storage_location}'
        COMMENT '{description}'
        """
        
        print(f"   📁 Storage location: {storage_location}")
        spark.sql(create_sql)
        created_catalogs.append(catalog_name)
        print(f"   ✅ Successfully created: {catalog_name}")
        
    except Exception as e:
        print(f"   ❌ Failed with managed location: {str(e)}")
        
        # Try Option 2: Without explicit location (let Databricks handle it)
        try:
            print(f"   🔄 Retrying without explicit storage location...")
            
            simple_create_sql = f"""
            CREATE CATALOG IF NOT EXISTS {catalog_name} 
            COMMENT '{description}'
            """
            
            spark.sql(simple_create_sql)
            created_catalogs.append(catalog_name)
            print(f"   ✅ Successfully created with default storage: {catalog_name}")
            
        except Exception as e2:
            failed_catalogs.append((catalog_name, str(e2)))
            print(f"   ❌ Also failed without location: {str(e2)}")
            
            # Check for specific error types
            if "INVALID_STATE" in str(e2) and "storage root URL" in str(e2):
                print(f"   💡 This is a Unity Catalog metastore configuration issue")
            elif "ACCESS_DENIED" in str(e2) or "PERMISSION" in str(e2).upper():
                print(f"   💡 This looks like a permissions issue")

# Summary
print(f"\n📊 CATALOG CREATION SUMMARY:")
print(f"   ✅ Successfully created: {len(created_catalogs)} catalogs")
if created_catalogs:
    for catalog in created_catalogs:
        print(f"      - {catalog}")

if failed_catalogs:
    print(f"   ❌ Failed to create: {len(failed_catalogs)} catalogs")
    for catalog, error in failed_catalogs:
        print(f"      - {catalog}")

# Provide solutions if catalogs failed
if failed_catalogs:
    print(f"\n💡 SOLUTIONS FOR FAILED CATALOGS:")
    
    print(f"\n🎯 OPTION 1: Use Databricks UI (EASIEST)")
    print(f"   1. Go to: https://adb-269274943657929.9.azuredatabricks.net/")
    print(f"   2. Navigate: Data → Create Catalog")
    print(f"   3. Create each catalog:")
    for catalog, _ in failed_catalogs:
        print(f"      - Name: {catalog}")
        print(f"      - Storage: Use Default Storage (recommended)")
    
    print(f"\n🔧 OPTION 2: Manual SQL with Different Paths")
    print(f"   Try these alternative create commands:")
    for catalog, _ in failed_catalogs:
        print(f"""
    -- Option 2A: With explicit DBFS path
    CREATE CATALOG IF NOT EXISTS {catalog} 
    MANAGED LOCATION 'dbfs:/user/hive/warehouse/{catalog}';
    
    -- Option 2B: With Unity Catalog volume path  
    CREATE CATALOG IF NOT EXISTS {catalog}
    MANAGED LOCATION 'abfss://unity-catalog@dbrootdl.dfs.core.windows.net/{catalog}';
        """)
    
    print(f"\n⚠️  OPTION 3: Ask Admin")
    print(f"   Contact your Databricks workspace admin to:")
    print(f"   - Configure Unity Catalog metastore properly")
    print(f"   - Grant you catalog creation permissions")
    print(f"   - Create these catalogs for you")

In [ ]:
# 🚨 QUICK FIX: Copy-Paste Commands for Your Current Notebook
print("🚨 IMMEDIATE SOLUTION - Copy these commands to your current notebook!")
print("=" * 70)

quick_fix_commands = """
-- 🎯 SOLUTION 1: Create catalogs with DBFS storage locations
CREATE CATALOG IF NOT EXISTS dev_digital_engineering_services 
MANAGED LOCATION 'dbfs:/mnt/unity-catalog/dev-storage'
COMMENT 'Development environment for MLOps pipeline';

CREATE CATALOG IF NOT EXISTS staging_digital_engineering_services 
MANAGED LOCATION 'dbfs:/mnt/unity-catalog/staging-storage'
COMMENT 'Staging environment for MLOps pipeline';

CREATE CATALOG IF NOT EXISTS prod_digital_engineering_services 
MANAGED LOCATION 'dbfs:/mnt/unity-catalog/prod-storage'
COMMENT 'Production environment for MLOps pipeline';

-- 🔍 Verify catalogs were created
SHOW CATALOGS;
"""

alternative_fix = """
-- 🎯 SOLUTION 2: Alternative shorter storage paths
CREATE CATALOG IF NOT EXISTS dev_digital_engineering_services 
MANAGED LOCATION 'dbfs:/user/hive/warehouse/dev_catalog'
COMMENT 'Development environment for MLOps pipeline';

CREATE CATALOG IF NOT EXISTS staging_digital_engineering_services 
MANAGED LOCATION 'dbfs:/user/hive/warehouse/staging_catalog'
COMMENT 'Staging environment for MLOps pipeline';

CREATE CATALOG IF NOT EXISTS prod_digital_engineering_services 
MANAGED LOCATION 'dbfs:/user/hive/warehouse/prod_catalog'
COMMENT 'Production environment for MLOps pipeline';

SHOW CATALOGS;
"""

print("📋 COPY THIS SQL TO YOUR CURRENT NOTEBOOK:")
print("=" * 50)
print(quick_fix_commands)

print("\n🔄 IF ABOVE FAILS, TRY THIS ALTERNATIVE:")
print("=" * 50)
print(alternative_fix)

print("\n💡 EASIEST SOLUTION - Use Databricks UI:")
print("   1. Open: https://adb-269274943657929.9.azuredatabricks.net/")
print("   2. Go to: Data → Create Catalog") 
print("   3. Create catalogs with 'Use Default Storage' option")

print("\n🎯 AFTER CREATING CATALOGS, RUN SCHEMAS:")
schema_commands = """
-- Then create the schemas:
CREATE SCHEMA IF NOT EXISTS dev_digital_engineering_services.default;
CREATE SCHEMA IF NOT EXISTS staging_digital_engineering_services.default;
CREATE SCHEMA IF NOT EXISTS prod_digital_engineering_services.default;

-- Verify everything worked:
SHOW SCHEMAS IN dev_digital_engineering_services;
"""
print(schema_commands)

print("🚀 Once catalogs are created, you can proceed with your MLOps pipeline!")

In [ ]:
# 🎉 GREAT NEWS: Some Catalogs Already Exist!
print("🎉 CHECKING EXISTING CATALOGS FROM YOUR WORKSPACE...")
print("=" * 60)

# Catalogs visible in your workspace
existing_catalogs = [
    "system",
    "dev_digital_engineering_services",  # ✅ This one exists!
    "dev_cdmsmith_shared",
    "postgres_edw_dev",
    "redshift_edw_dev",
    "samples",
    "hive_metastore"
]

required_catalogs = [
    "dev_digital_engineering_services",
    "staging_digital_engineering_services", 
    "prod_digital_engineering_services"
]

print("📋 CATALOG STATUS:")
available_for_mlops = []
missing_for_mlops = []

for catalog in required_catalogs:
    if catalog in existing_catalogs:
        print(f"   ✅ {catalog} - EXISTS!")
        available_for_mlops.append(catalog)
    else:
        print(f"   ❌ {catalog} - MISSING")
        missing_for_mlops.append(catalog)

print(f"\n🎯 RECOMMENDED APPROACH:")
if available_for_mlops:
    print(f"   Use existing catalog: {available_for_mlops[0]}")
    print(f"   Create schemas for all environments in this catalog:")
    
    main_catalog = available_for_mlops[0]
    schemas_to_create = [
        f"{main_catalog}.default",
        f"{main_catalog}.staging", 
        f"{main_catalog}.prod"
    ]
    
    print(f"\n📝 SCHEMA STRUCTURE TO CREATE:")
    for schema in schemas_to_create:
        print(f"   📊 {schema}")
    
    print(f"\n🚀 NEXT STEP: Run the schema creation SQL below!")
else:
    print(f"   ❌ No MLOps catalogs found - contact your admin")

print(f"\n💡 Alternative: You could also use 'hive_metastore' if admin permissions are limited")

In [ ]:
# 🚀 READY-TO-RUN: Create Schemas in Your Existing Catalog
print("🚀 CREATING SCHEMAS IN EXISTING CATALOG...")
print("=" * 60)

# Use the existing dev_digital_engineering_services catalog
main_catalog = "dev_digital_engineering_services"

print(f"📊 Using existing catalog: {main_catalog}")
print(f"🎯 Creating schemas for DEV, STAGING, and PROD environments")

# Create schemas for different environments
schemas_config = [
    ("default", "Default schema for development environment"),
    ("staging", "Schema for staging environment"), 
    ("prod", "Schema for production environment")
]

created_schemas = []
failed_schemas = []

for schema_name, description in schemas_config:
    try:
        print(f"\n🚀 Creating: {main_catalog}.{schema_name}")
        
        create_schema_sql = f"""
        CREATE SCHEMA IF NOT EXISTS {main_catalog}.{schema_name}
        COMMENT '{description}'
        """
        
        # Execute the SQL
        result = spark.sql(create_schema_sql)
        created_schemas.append(f"{main_catalog}.{schema_name}")
        print(f"   ✅ SUCCESS: {main_catalog}.{schema_name}")
        
    except Exception as e:
        failed_schemas.append((f"{main_catalog}.{schema_name}", str(e)))
        print(f"   ❌ Failed: {str(e)}")

# Verification
print(f"\n📊 SCHEMA CREATION RESULTS:")
print(f"   ✅ Created: {len(created_schemas)} schemas")
for schema in created_schemas:
    print(f"      - {schema}")

if failed_schemas:
    print(f"   ❌ Failed: {len(failed_schemas)} schemas")
    for schema, error in failed_schemas:
        print(f"      - {schema}: {error}")

# Show the schemas that were created
if created_schemas:
    print(f"\n🔍 VERIFYING SCHEMAS:")
    try:
        schemas_df = spark.sql(f"SHOW SCHEMAS IN {main_catalog}")
        display(schemas_df)
        
        print(f"\n🎉 SUCCESS! Your MLOps pipeline can now use:")
        for schema in created_schemas:
            print(f"   📊 {schema}")
            
        print(f"\n📝 UPDATE YOUR CONFIG:")
        print(f"   MODEL_NAME = '{main_catalog}.default.water_quality_model'")
        print(f"   FEATURE_TABLE = '{main_catalog}.default.water_quality_features'")
        print(f"   \n   For staging: {main_catalog}.staging")
        print(f"   For prod: {main_catalog}.prod")
        
    except Exception as e:
        print(f"❌ Could not verify schemas: {str(e)}")

print(f"\n🚀 READY TO PROCEED! Your Unity Catalog setup is complete!")

In [ ]:
# 📊 Create Schemas with Error Handling
print("📊 Creating schemas in available catalogs...")
print("=" * 60)

# Define schemas to create
schemas_to_create = [
    ("dev_digital_engineering_services", "default", "Development schema for MLOps pipeline"),
    ("staging_digital_engineering_services", "default", "Staging schema for MLOps pipeline"),
    ("prod_digital_engineering_services", "default", "Production schema for MLOps pipeline")
]

# First, get current available catalogs
current_catalogs = []
try:
    current_catalogs = [row['catalog'] for row in spark.sql("SHOW CATALOGS").collect()]
except:
    print("⚠️  Could not fetch current catalogs list")

created_schemas = []
failed_schemas = []
alternative_suggestions = []

for catalog_name, schema_name, description in schemas_to_create:
    try:
        print(f"\n🚀 Creating schema: {catalog_name}.{schema_name}")
        
        # Check if catalog exists first
        if catalog_name not in current_catalogs:
            print(f"   ⚠️  Catalog {catalog_name} not found")
            failed_schemas.append((f"{catalog_name}.{schema_name}", f"Catalog {catalog_name} does not exist"))
            continue
        
        # Create schema
        create_schema_sql = f"""
        CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}
        COMMENT '{description}'
        """
        
        spark.sql(create_schema_sql)
        created_schemas.append(f"{catalog_name}.{schema_name}")
        print(f"   ✅ Successfully created: {catalog_name}.{schema_name}")
        
    except AnalysisException as e:
        if "NO_SUCH_CATALOG_EXCEPTION" in str(e):
            failed_schemas.append((f"{catalog_name}.{schema_name}", f"Catalog {catalog_name} not found"))
            print(f"   ❌ Catalog not found: {catalog_name}")
            
            # Suggest using available catalogs
            if current_catalogs:
                available_alt = current_catalogs[0]  # Use first available catalog
                alternative_suggestions.append(f"   💡 Alternative: Use {available_alt}.mlops_dev, {available_alt}.mlops_staging, {available_alt}.mlops_prod")
        else:
            failed_schemas.append((f"{catalog_name}.{schema_name}", str(e)))
            print(f"   ❌ Failed: {str(e)}")
    except Exception as e:
        failed_schemas.append((f"{catalog_name}.{schema_name}", str(e)))
        print(f"   ❌ Unexpected error: {str(e)}")

# Summary
print(f"\n📊 SCHEMA CREATION SUMMARY:")
print(f"   ✅ Successfully created: {len(created_schemas)} schemas")
for schema in created_schemas:
    print(f"      - {schema}")

if failed_schemas:
    print(f"   ❌ Failed to create: {len(failed_schemas)} schemas")
    for schema, error in failed_schemas:
        print(f"      - {schema}: {error}")

# Show alternatives if some failed
if failed_schemas and current_catalogs:
    print(f"\n💡 ALTERNATIVE APPROACH - Use Single Catalog:")
    print(f"   If you can't create multiple catalogs, use one existing catalog:")
    if current_catalogs:
        main_catalog = current_catalogs[0]
        print(f"   📋 Available catalog to use: {main_catalog}")
        print(f"")
        print(f"   🔄 Alternative schema structure:")
        print(f"      - {main_catalog}.mlops_dev     (instead of dev_digital_engineering_services.default)")
        print(f"      - {main_catalog}.mlops_staging (instead of staging_digital_engineering_services.default)")
        print(f"      - {main_catalog}.mlops_prod    (instead of prod_digital_engineering_services.default)")
        print(f"")
        print(f"   📝 Run this to create alternative schemas:")
        print(f"""
        CREATE SCHEMA IF NOT EXISTS {main_catalog}.mlops_dev COMMENT 'MLOps Dev Environment';
        CREATE SCHEMA IF NOT EXISTS {main_catalog}.mlops_staging COMMENT 'MLOps Staging Environment';
        CREATE SCHEMA IF NOT EXISTS {main_catalog}.mlops_prod COMMENT 'MLOps Production Environment';
        """)

In [ ]:
# 🔄 Alternative: Create Schemas Using Single Catalog (If Above Failed)
print("🔄 Creating alternative schema structure using existing catalog...")
print("=" * 60)

# Get first available catalog for alternative approach
try:
    available_catalogs = [row['catalog'] for row in spark.sql("SHOW CATALOGS").collect()]
    
    if available_catalogs:
        main_catalog = available_catalogs[0]
        print(f"📋 Using catalog: {main_catalog}")
        
        # Alternative schema structure
        alt_schemas = [
            ("mlops_dev", "Development environment for MLOps pipeline"),
            ("mlops_staging", "Staging environment for MLOps pipeline"),
            ("mlops_prod", "Production environment for MLOps pipeline")
        ]
        
        alt_created = []
        alt_failed = []
        
        for schema_name, description in alt_schemas:
            try:
                print(f"\n🚀 Creating: {main_catalog}.{schema_name}")
                
                create_alt_sql = f"""
                CREATE SCHEMA IF NOT EXISTS {main_catalog}.{schema_name}
                COMMENT '{description}'
                """
                
                spark.sql(create_alt_sql)
                alt_created.append(f"{main_catalog}.{schema_name}")
                print(f"   ✅ Success: {main_catalog}.{schema_name}")
                
            except Exception as e:
                alt_failed.append((f"{main_catalog}.{schema_name}", str(e)))
                print(f"   ❌ Failed: {str(e)}")
        
        # Update configuration recommendations
        if alt_created:
            print(f"\n🎉 SUCCESS! Created {len(alt_created)} alternative schemas:")
            for schema in alt_created:
                print(f"   ✅ {schema}")
            
            print(f"\n📝 UPDATE YOUR CONFIG.YAML:")
            print(f"   Replace the catalog configurations with:")
            print(f"""
            catalog:
              dev: {main_catalog}
              staging: {main_catalog} 
              prod: {main_catalog}
            
            # And update schema names in your code:
            # DEV: {main_catalog}.mlops_dev
            # STAGING: {main_catalog}.mlops_staging  
            # PROD: {main_catalog}.mlops_prod
            """)
            
            print(f"\n🔧 UPDATE MODEL NAMES IN NOTEBOOK:")
            print(f"   Change MODEL_NAME to:")
            print(f"   MODEL_NAME = '{main_catalog}.mlops_dev.water_quality_model'")
            print(f"   FEATURE_TABLE = '{main_catalog}.mlops_dev.water_quality_features'")
                
    else:
        print("❌ No catalogs available - please contact your Databricks admin")
        
except Exception as e:
    print(f"❌ Error with alternative approach: {str(e)}")
    print("💡 You may need to manually create schemas or contact admin")

In [ ]:
# ✅ Verify Schema Creation & Final Status
print("✅ FINAL VERIFICATION - Checking all schemas...")
print("=" * 60)

# Check all catalogs and their schemas
try:
    print(f"🔍 Current catalog structure:")
    catalogs = spark.sql("SHOW CATALOGS").collect()
    
    total_schemas_found = 0
    mlops_schemas = []
    
    for catalog_row in catalogs:
        catalog_name = catalog_row['catalog']
        print(f"\n📁 Catalog: {catalog_name}")
        
        try:
            # Show schemas in this catalog
            schemas = spark.sql(f"SHOW SCHEMAS IN {catalog_name}").collect()
            
            for schema_row in schemas:
                schema_name = schema_row['databaseName']
                full_name = f"{catalog_name}.{schema_name}"
                print(f"   📊 {full_name}")
                total_schemas_found += 1
                
                # Track MLOps-related schemas
                if any(keyword in schema_name.lower() for keyword in ['mlops', 'digital_engineering', 'default']):
                    mlops_schemas.append(full_name)
                    
        except Exception as e:
            print(f"   ❌ Could not list schemas: {str(e)}")
    
    print(f"\n📊 SUMMARY:")
    print(f"   📁 Total catalogs: {len(catalogs)}")
    print(f"   📊 Total schemas found: {total_schemas_found}")
    print(f"   🎯 MLOps-related schemas: {len(mlops_schemas)}")
    
    if mlops_schemas:
        print(f"\n🎉 AVAILABLE SCHEMAS FOR YOUR MLOPS PIPELINE:")
        for i, schema in enumerate(mlops_schemas, 1):
            print(f"   {i}. {schema}")
            
        # Provide next steps
        print(f"\n🚀 NEXT STEPS:")
        print(f"   1. ✅ Schemas are ready!")
        print(f"   2. 📝 Update your config.yaml with the correct catalog names")
        print(f"   3. 🚀 Run the main MLOps notebook: notebooks/main_nb.ipynb")
        print(f"   4. 📊 Upload your water_potability.csv data")
        
        # Show example configuration
        if mlops_schemas:
            first_schema = mlops_schemas[0]
            catalog_part = first_schema.split('.')[0]
            print(f"\n📝 EXAMPLE CONFIG UPDATE:")
            print(f"   MODEL_NAME = '{first_schema}.water_quality_model'")
            print(f"   FEATURE_TABLE = '{first_schema}.water_quality_features'")
            
    else:
        print(f"\n⚠️  No MLOps schemas found. You may need to:")
        print(f"   1. Create schemas manually")
        print(f"   2. Check permissions")
        print(f"   3. Contact your Databricks admin")
        
except Exception as e:
    print(f"❌ Error during verification: {str(e)}")

print(f"\n🎊 SETUP COMPLETE! You can now proceed with your MLOps pipeline!")

## 🚨 **Troubleshooting Guide**

### **Common Errors & Solutions:**

#### **❌ Error: NO_SUCH_CATALOG_EXCEPTION**
```
[NO_SUCH_CATALOG_EXCEPTION] Catalog 'staging_digital_engineering_services' was not found
```
**💡 Solutions:**
1. **Run cells 2-4 above** to create missing catalogs
2. **Use alternative single-catalog approach** (cell 5)
3. **Contact admin** to create catalogs for you

#### **❌ Error: ACCESS_DENIED or PERMISSION_DENIED**
**💡 Solutions:**
1. Ask admin for **catalog creation permissions**  
2. Use **existing catalogs** instead of creating new ones
3. Request access to **Unity Catalog metastore**

#### **❌ Error: Cannot create schema**
**💡 Solutions:**
1. Check if **catalog exists** first (run cell 2)
2. Verify you have **schema creation** permissions
3. Try the **single catalog approach** (cell 5)

---

### **🎯 Next Steps After Success:**

1. **✅ Update Configuration Files:**
   - Update `configs/config.yaml` with correct catalog names
   - Update `MODEL_NAME` in your notebooks

2. **📊 Upload Data:**
   - Upload `water_potability.csv` to Databricks
   - Place in appropriate Volume path

3. **🚀 Run Main Pipeline:**
   - Open `notebooks/main_nb.ipynb`
   - Execute cells sequentially
   - Monitor model training and inference

4. **🤖 Setup CI/CD:**
   - Configure GitHub secrets
   - Enable automated workflows

---

### **📋 Quick Commands Reference:**

```sql
-- Check available catalogs
SHOW CATALOGS;

-- Check schemas in a catalog
SHOW SCHEMAS IN your_catalog_name;

-- Create catalog (if permissions allow)
CREATE CATALOG IF NOT EXISTS your_catalog_name;

-- Create schema
CREATE SCHEMA IF NOT EXISTS your_catalog.your_schema;
```

**🎊 You're all set! Proceed with running your enhanced MLOps notebook!**